[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/vortices/vortices.ipynb)

# Leapfrogging Vortices

This example follows vortices in a plane of fluid and the derivative of the flow they make. Around each vortex the fluid turns, fast near its core and slower further out, and the vortices carry one another along. Two pairs travelling side by side take turns passing through each other, leapfrogging; with the inner pair too narrow, the leapfrog gives way to a lap, the walkabout; and traced through time, each vortex draws a worldline, and the leapfrogging pairs braid. The derivative of the flow tells where the fluid turns and where it is only strained.

Each vortex carries its spin, a bivector: the turn it gives the fluid, its size the circulation and its sign the sense of the turn. A point moves with its commutator with the spin, `separation.commutator(rate)`, so the sense comes from the vortices themselves, and a mirror image of the vortices, every spin turned the other way, moves as the mirror image of their motion. The velocity gradient at a point is a map on displacements, `Point <- Point`, written in closed form as a sum over the vortices. The derivative of the flow is that map multiplied by `Point`, the space of all points, and contracted, `(Point * gradient(Point)).contract(1, 2)`: its scalar part is the divergence and its bivector part the vorticity. The trace of the gradient applied twice, `-0.5 * gradient(gradient).trace()`, needs no metric and weighs swirl against strain. A ring of vortices that turns as one shows it in the algebra directly: each vortex's place times its first place is the same rotor for all of them.

In complex notation, with the velocity $(u, v)$ written as $u - iv$, point vortices of circulation $\Gamma_k$ at $z_k$ move the fluid at $u - iv = \sum_k \frac{\Gamma_k}{2\pi i\,(z - z_k)}$, which the vortex blob softens within its core.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from dataclasses import dataclass, replace

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA2D as ga
from examples.util.animation import show_animation
from examples.fluids.vortices import render
from examples.fluids.vortices.core import grid

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Points, and displacements and velocities of points, are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()
# A vortex's spin: the turn it gives the fluid, its size the circulation.
Spin = ga.gatype.bivector()
Even = ga.gatype.even()
Gradient = ga.gatype((Point, Point))                                         # Point <- Point

## 1. The flow around the vortices

Each vortex turns the fluid around its centre, and its spin says which way and how strongly: the circulation times the unit bivector of the plane, `mv.xy * circulations`. A separation `r` from the centre, turned a quarter turn by the spin, is the commutator `r.commutator(spin)`, and the vortex moves the fluid along it. How fast follows the vortex blob of Rosenhead and Moore: near the centre the fluid turns like a solid disc, and beyond the core the speed falls off as one over the distance. The velocity at a point is the sum over the vortices.

In [ ]:
# An outer and an inner pair on one axis, each pair turning in opposite senses so that it travels
# along x, one entry per vortex, and the core of each.
half_spans = np.array([1.0, -1.0, 0.55, -0.55])
circulations = np.array([1.0, -1.0, 1.0, -1.0])
core_radius = 0.15
# The pattern does not repeat: its only copy is itself.
alone = mv.vector([[0.0, 0.0]]).dual()                                       # [copies] Point
# The window around the vortices, and its samples.
half_width, half_height = 2.4, 1.6
n_columns, n_rows = 300, 200

In [ ]:
@dataclass(frozen=True)
class Vortices:
    centres: Point                                           # [..., vortices] Point
    spins: Spin                                              # [vortices] Spin
    core_radius: float
    copies: Point                                            # [copies] Point, offsets the pattern repeats at


def separations(vortices: Vortices, points: Point) -> Point:
    """From every vortex and each of its copies to every point."""
    # Two new axes on the points pair every point with every vortex and every copy by broadcasting.
    return points[..., None, None] - (vortices.centres[..., None] + vortices.copies)   # [..., vortices, copies] Point


def profile(vortices: Vortices, separations: Point) -> tuple[Spin, Spin]:
    """The turn the fluid makes about each centre, and its change with the squared distance."""
    # The squared distance, softened by the core: finite at the centre, a point vortex far from it.
    blob = (separations | separations) + vortices.core_radius**2                  # [..., vortices, copies] Scalar
    # The circulation spread around a circle through the point gives the speed there; divided once
    # more by the distance, the rate at which the fluid turns, in the sense of the vortex's spin.
    rate = vortices.spins[:, None] / (2 * np.pi) / blob                          # [..., vortices, copies] Spin
    return rate, -rate / blob


def velocity(vortices: Vortices, points: Point) -> Point:
    offsets = separations(vortices, points)                                       # [..., vortices, copies] Point
    rate, _ = profile(vortices, offsets)                                          # [..., vortices, copies] Spin
    # The commutator turns each separation a quarter turn by the spin and scales it by the rate: the
    # fluid's velocity around that vortex, summed over the vortices and their copies.
    return offsets.commutator(rate).sum(axis=-1).sum(axis=-1)                     # [...] Point


# The points on the axis hold one component only; the cast embeds them in the plane, where the vortices will move.
vortices = Vortices((mv.y * half_spans).dual().cast(Point), mv.xy * circulations, core_radius, alone)
# A window around the vortices, sampled at the centres of its cells.
points = vortices.centres.mean(axis=-1) + grid(half_width, half_height, n_columns, n_rows)   # [rows, columns] Point
flow = velocity(vortices, points)                                                 # [rows, columns] Point

## 2. The derivative of the flow

The velocity gradient at a point is a map on displacements, `Point <- Point`: the change in velocity over a small step. A step turns with the rate at the point, `Point.commutator(rate)`, and it also changes the rate by its component along the separation, `separation | Point`, which acts on the turned separation. Multiplied by `Point`, the space of all points, and contracted, `(Point * gradient(Point)).contract(1, 2)`, the gradient's output is paired with its input through the metric: this is the derivative of the flow. Its scalar part is the divergence, which vanishes here, as the fluid neither gathers nor spreads. Its bivector part is the vorticity, the rate at which the fluid turns, concentrated in the cores. Away from them the vorticity falls off as the fourth power of the distance and the whole derivative all but vanishes: there the flow nearly meets the Cauchy–Riemann condition, the mark of an analytic flow, which point vortices without cores would meet exactly.

In the notation of vector calculus the scalar part reads as $\nabla\cdot\mathbf v$ and the bivector part as $(\partial_x v_y - \partial_y v_x)\,\mathbf e_x\wedge\mathbf e_y$.

In the notation of complex analysis, with the velocity $\mathbf v = u\,\mathbf e_x + v\,\mathbf e_y$ written as $u - iv$, the vanishing derivative reads as the Cauchy–Riemann equations $\partial_x u + \partial_y v = 0$ and $\partial_y u - \partial_x v = 0$.

In [ ]:
def gradient(vortices: Vortices, points: Point) -> Gradient:
    offsets = separations(vortices, points)                                       # [..., vortices, copies] Point
    rate, rate_change = profile(vortices, offsets)                                # [..., vortices, copies] Spin each
    # Over `Point`, the space of all displacements, each term is a map from a small step to the change in
    # velocity. Turning: the step is carried around the vortex like the separation itself.
    turning = Point.commutator(rate)                                              # [..., vortices, copies] Point <- Point
    # Spreading: the part of the step along the separation changes the distance, and so the rate.
    spreading = 2 * offsets.commutator(rate_change) * (offsets | Point)           # [..., vortices, copies] Point <- Point
    return (turning + spreading).sum(axis=-1).sum(axis=-1)                        # [...] Point <- Point


def derivative(gradients: Gradient) -> Even:
    # `Point`, the space of all points, multiplied into the gradient's output and paired with its input
    # through the metric: each basis point times the gradient of its reciprocal, summed, without naming a basis.
    return (Point * gradients(Point)).contract(1, 2)                              # [...] Even


gradients = gradient(vortices, points)                                            # [rows, columns] Point <- Point
derivatives = derivative(gradients)                                               # [rows, columns] Even
# The point midway between the two vortices above the axis.
between = (vortices.centres[0] + vortices.centres[2]) / 2                         # [] Point
between_gradient = gradient(vortices, between)                                    # [] Point <- Point
between_derivative = derivative(between_gradient)                                 # [] Even

Printed below, the gradient's coefficients at that point are not symmetric. The derivative there has no scalar part, and its vorticity is the difference of the two off-diagonal coefficients, the rate at which the fluid turns.

In [ ]:
print(f"gradient\n{between_gradient.coefficient_table()}\n")
print(f"derivative\n{between_derivative.coefficient_table()}")

## 3. Swirl against strain

The gradient's skew part turns the fluid and its symmetric part stretches it. The trace of the gradient applied twice pairs its output with its input and needs no metric, and it weighs the two against each other: `-0.5 * gradient(gradient).trace()` is positive where the fluid turns faster than it stretches, inside the cores, and negative where the pairs pull the fluid between them apart. For a flow without divergence in the plane it equals the gradient's determinant. Midway between the upper vortices it is negative: the mean of the off-diagonal coefficients, the stretch, outweighs the turn.

In the tensor notation of the velocity gradient the swirl reads as the Q-criterion, $Q = \tfrac12\left(\lVert\Omega\rVert^2 - \lVert S\rVert^2\right)$, with $\Omega$ and $S$ the skew and symmetric parts of the velocity gradient.

In [ ]:
# The colour scales of the vorticity and the swirl.
vorticity_limit, swirl_limit = 8.0, 6.0

In [ ]:
def swirl(gradients: Gradient) -> Scalar:
    return -0.5 * gradients(gradients).trace()                                    # [...] Scalar


swirls = swirl(gradients)                                                         # [rows, columns] Scalar
between_swirl = swirl(between_gradient)                                           # [] Scalar
render.figure(points, derivatives, swirls, vorticity_limit, swirl_limit);

## 4. Leapfrogging

Each vortex moves with the flow at its centre, to which its own blob adds nothing, stepped by the classical fourth-order Runge–Kutta rule. The inner pair, narrower, travels faster and slips through the outer pair, which widens it and slows it down; the outer pair narrows, speeds up, and passes in its turn.

In [ ]:
# The step, and the steps the pairs travel before the second still.
dt = 0.12
n_steps = 100

In [ ]:
def drift(vortices: Vortices) -> Point:
    """The velocity of each vortex: the flow at its centre."""
    # Each vortex moves with the others' flow; its own blob is still at its centre, where the
    # separation vanishes.
    return velocity(vortices, vortices.centres)                                   # [vortices] Point


def step(vortices: Vortices, dt: float) -> Vortices:
    def moved(rate: Point, fraction: float) -> Vortices:
        return replace(vortices, centres=vortices.centres + rate * (fraction * dt))

    # Four looks at the drift across the step: at its start, twice halfway along the previous look,
    # and at its end; averaged with weights one, two, two, one.
    first = drift(vortices)                                                       # [vortices] Point
    second = drift(moved(first, 0.5))                                             # [vortices] Point
    third = drift(moved(second, 0.5))                                             # [vortices] Point
    fourth = drift(moved(third, 1.0))                                             # [vortices] Point
    return moved((first + 2 * second + 2 * third + fourth) / 6, 1.0)


# The same panels after the pairs have passed each other a few times.
travelled = vortices
for _ in range(n_steps):
    travelled = step(travelled, dt)
travelled_points = travelled.centres.mean(axis=-1) + grid(half_width, half_height, n_columns, n_rows)   # [rows, columns] Point
travelled_gradients = gradient(travelled, travelled_points)                       # [rows, columns] Point <- Point
render.figure(travelled_points, derivative(travelled_gradients), swirl(travelled_gradients), vorticity_limit, swirl_limit);

In [ ]:
# The animation: each frame's duration, its frames and the steps between them. The cores are smooth,
# so a grid much coarser than the stills' shows them as well and keeps pace.
duration_ms = 40
n_frames, n_substeps = 120, 2
samples_per_unit = 8
# One strip around the whole trip, so that each vortex's trail stays in view.
trip_centre = mv.x.dual() * 3.0                                                   # [] Point
trip_half_width, trip_half_height = 3.6, 1.4

In [ ]:
def evolve(vortices: Vortices, points: Point, n_frames: int, n_substeps: int, dt: float) -> Iterator[tuple[Vortices, Even]]:
    """The vortices, and the derivative of their flow on the points, at each frame, a few steps apart."""
    for _ in range(n_frames):
        yield vortices, derivative(gradient(vortices, points))
        for _ in range(n_substeps):
            vortices = step(vortices, dt)


def strip(centre: Point, half_width: float, half_height: float, samples_per_unit: float) -> Point:
    """A window around a centre, sampled evenly."""
    return centre + grid(half_width, half_height, round(2 * half_width * samples_per_unit), round(2 * half_height * samples_per_unit))


trip = strip(trip_centre, trip_half_width, trip_half_height, samples_per_unit)    # [rows, columns] Point
show_animation(render.window(trip, evolve(vortices, trip, n_frames, n_substeps, dt), vorticity_limit), "vortices_leapfrog", duration_ms)

## 5. Walkabout

Leapfrogging is stable only while the inner pair is wider than about 0.38 of the outer, one over the square of the golden ratio. Below that, a nudge that breaks the mirror symmetry grows. Between about 0.29 and 0.38 the four vortices do not fly apart: the leapfrog gives way to a lap in which the two vortices of one sense close into a tight pair, the other two circle it, one close and one wide, and the group stalls; then the pair opens up again and the leapfrog carries on. This is the walkabout that Acheson found; Behring and Goodman derived the threshold.

In [ ]:
# The inner pair at 0.35 of the outer, where leapfrogging is unstable, and one inner vortex nudged
# along x to break the mirror symmetry: without the nudge, the symmetric leapfrog carries on.
walkabout_half_spans = np.array([1.0, -1.0, 0.35, -0.35])
nudges = np.array([0.0, 0.0, 0.015, 0.0])
# A longer step than the leapfrog's traces the same lap. The strip is centred on the lap, and the run goes on
# past it as far as it leapfrogged before it.
walkabout_dt = 0.2
n_walkabout_frames, n_walkabout_substeps = 135, 5
walkabout_centre = (mv.x * 10.0 + mv.y * 1.4).dual()                              # [] Point
walkabout_half_width, walkabout_half_height = 10.6, 2.9

In [ ]:
walker = Vortices((mv.y * walkabout_half_spans + mv.x * nudges).dual(), mv.xy * circulations, core_radius, alone)
walk = strip(walkabout_centre, walkabout_half_width, walkabout_half_height, samples_per_unit)   # [rows, columns] Point
show_animation(render.window(walk, evolve(walker, walk, n_walkabout_frames, n_walkabout_substeps, walkabout_dt), vorticity_limit), "vortices_walkabout", duration_ms)

## 6. Worldlines

A worldline is the curve a vortex traces through the plane and time together, drawn here with time running upward. Five equal vortices evenly spaced on a ring turn rigidly about its centre, each carried along the ring by the other four, so their worldlines are five helices wound around one vertical axis. The leapfrogging pairs, drawn following their mean position as they travel along `x`, braid: the two vortices of one sense, inner and outer, wind around each other, one full turn for every two passes. Time has no scale of its own against distance, so each braid's drawn time is stretched to a fixed multiple of its width. Each colour is one vortex.

The ring's places are one place turned by rotors, `turns >> (mv.x.dual() * ring_radius)`, and the ring keeps turning as one: each vortex's place times its first place, `ring_worldlines * ring_worldlines[0]`, a rotor scaled by the squared radius, is the same for all five at every step.

In [ ]:
# Five equal vortices evenly spaced on a ring of unit radius around the origin, with a small core;
# the step, the steps for one and a half turns of the ring and three periods of the leapfrog, and the
# height of each braid's drawn time against its width.
n_ring, ring_radius, ring_core = 5, 1.0, 0.05
ring_circulations = np.ones(n_ring)
worldline_dt = 0.2
n_ring_steps, n_leapfrog_steps = 150, 75
braid_height = 2.5

In [ ]:
def worldlines(vortices: Vortices, dt: float, n_steps: int) -> Point:
    """The centres of the vortices at every step: one curve through the plane and time per vortex."""
    def stepped() -> Iterator[Point]:
        current = vortices
        yield current.centres
        for _ in range(n_steps):
            current = step(current, dt)
            yield current.centres

    # The steps stacked along a new leading axis: time.
    return stack(list(stepped()))                                                  # [steps + 1, vortices] Point


# A rotor turns by twice the angle in its exponent: these turn the first place to each of the others.
turns = ((mv.x ^ mv.y) * (-np.pi * np.arange(n_ring) / n_ring)).exp()        # [vortices] Rotor
ring = Vortices(turns >> (mv.x.dual() * ring_radius), mv.xy * ring_circulations, ring_core, alone)
ring_worldlines = worldlines(ring, worldline_dt, n_ring_steps)                       # [steps + 1, vortices] Point
leapfrog_worldlines = worldlines(vortices, worldline_dt, n_leapfrog_steps)           # [steps + 1, vortices] Point
render.braids([ring_worldlines, leapfrog_worldlines], braid_height);

In [ ]:
# checks
# The gradient is the change in velocity over a small displacement.
displacement = mv.vector([0.3, -0.2]).dual() * 1e-5                               # [] Point
change = (velocity(vortices, points + displacement) - velocity(vortices, points - displacement)) / 2
np.testing.assert_allclose((gradients(displacement) - change).kernel, 0.0, atol=1e-12)
# The derivative has no divergence, and its bivector is the blobs' vorticity.
blob = (separations(vortices, points) | separations(vortices, points)) + core_radius**2   # [rows, columns, vortices, copies] Scalar
vorticity = (vortices.spins[:, None] * core_radius**2 / np.pi / (blob * blob)).sum(axis=-1).sum(axis=-1)   # [rows, columns] Spin
np.testing.assert_allclose((derivatives - vorticity).kernel, 0.0, atol=1e-10)
# Without divergence, swirl is the gradient's determinant, negative between the upper vortices.
np.testing.assert_allclose((swirls - gradients.det()).kernel, 0.0, atol=1e-10)
assert between_swirl.kernel[0] < 0
# Along both runs of worldlines the linear impulse, the circulation-weighted sum of the centres, stays
# the same to round-off; the ring turns as one: each place times its first place is the same for all.
for weights, centres in ((ring_circulations, ring_worldlines), (circulations, leapfrog_worldlines)):
    linear = (weights * centres).sum(axis=-1)                                      # [steps + 1] Point
    np.testing.assert_allclose((linear - linear[0]).kernel, 0.0, atol=1e-12)
turned = ring_worldlines * ring_worldlines[0]                                      # [steps + 1, vortices] Even
np.testing.assert_allclose((turned - turned[:, :1]).kernel, 0.0, atol=1e-12)